# ai-therapist SFT — Qwen3-4B-Instruct (Unsloth QLoRA)
Non-thinking conversational finetune on `data/finetune/counsel-chat` (+ `data/finetune/amod-counseling` when present). Run top-to-bottom on free Colab T4 or local 8GB GPU.


### Installation

In [ ]:
%%capture
import os, re

if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch

    v = re.match(r"[\d]{1,}\.[\d]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + {"2.9": "0.0.33.post1", "2.8": "0.0.32.post2"}.get(
        v, "0.0.35"
    )
    if str(torch.version.cuda).startswith("13") and xformers.endswith("0.0.35"):
        xformers = "https://download.pytorch.org/whl/cu130/xformers-0.0.35-py39-none-manylinux_2_28_x86_64.whl"
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

### Load model

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-4B-Instruct-2507",
    max_seq_length=2048,
    load_in_4bit=True,
    load_in_8bit=False,
    full_finetuning=False,
)

LoRA adapters — only a small % of params train.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

### Data — local `data/` (CounselChat + Amod)

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from datasets import load_dataset, concatenate_datasets
import glob

tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")


def to_convos(ds, user_col, asst_col):
    ds = ds.filter(
        lambda x: bool(str(x[user_col]).strip()) and bool(str(x[asst_col]).strip())
    )

    def _map(ex):
        return {
            "conversations": [
                {"role": "user", "content": str(ex[user_col]).strip()},
                {"role": "assistant", "content": str(ex[asst_col]).strip()},
            ]
        }

    drop = [c for c in ds.column_names if c not in (user_col, asst_col)]
    return ds.map(_map, remove_columns=drop)


# Both datasets required — combined + shuffled (single-stage SFT, not sequential)
cc = glob.glob("data/finetune/counsel-chat/*.csv")
assert cc, "missing data/finetune/counsel-chat/*.csv"
counsel = to_convos(
    load_dataset("csv", data_files=cc[0], split="train"), "questionText", "answerText"
)

amod = glob.glob("data/finetune/amod-counseling/*.csv") + glob.glob(
    "data/finetune/amod-counseling/*.parquet"
)
assert amod, "missing data/finetune/amod-counseling/* (accept gated terms first)"
fmt = "csv" if amod[0].endswith(".csv") else "parquet"
a = load_dataset(fmt, data_files=amod[0], split="train")
u, r = (
    ("Context", "Response")
    if "Context" in a.column_names
    else (a.column_names[0], a.column_names[1])
)
amod_ds = to_convos(a, u, r)

dataset = concatenate_datasets([counsel, amod_ds]).shuffle(seed=3407)
dataset = standardize_data_formats(dataset)
print(len(dataset), "examples (counsel:", len(counsel), "+ amod:", len(amod_ds), ")")
print(dataset[0]["conversations"][0]["content"][:200])

Apply Qwen3 chat template → `text`.

In [ ]:
def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}


dataset = dataset.map(formatting_prompts_func, batched=True)
print(dataset[0]["text"][:300])

### Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=1,
        learning_rate=2e-4,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        report_to="none",
    ),
)
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(trainer)
trainer_stats = trainer.train()

### Sanity-check inference (non-thinking: temp 0.7 / top_p 0.8)

In [ ]:
messages = [
    {
        "role": "user",
        "content": "I have been feeling anxious about work lately and I can't sleep. What should I do?",
    }
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
from transformers import TextStreamer

_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True)
)

### Save LoRA adapters (+ optional GGUF for Ollama)

In [ ]:
import os

try:
    from dotenv import load_dotenv

    load_dotenv()
except ImportError:
    import subprocess

    subprocess.run(["pip", "install", "python-dotenv"], check=True)
    from dotenv import load_dotenv

    load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")  # set in .env
assert HF_TOKEN, "set HF_TOKEN in .env"

model.save_pretrained("qwen_lora")
tokenizer.save_pretrained("qwen_lora")
model.push_to_hub("arnabwithab/ai-therapist-qwen3-4b-lora", token=HF_TOKEN)
tokenizer.push_to_hub("arnabwithab/ai-therapist-qwen3-4b-lora", token=HF_TOKEN)
# GGUF q4_k_m for Ollama/llama.cpp:
model.save_pretrained_gguf("qwen_finetune", tokenizer, quantization_method="q4_k_m")
model.push_to_hub_gguf(
    "arnabwithab/ai-therapist-qwen3-4b-gguf",
    tokenizer,
    quantization_method="q4_k_m",
    token=HF_TOKEN,
)